<a href="https://colab.research.google.com/github/jacobwechuli/QA-intelligent-RAG-system/blob/main/QA_intelligent_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q gradio
!pip install -q gradio_pdf
!pip install -q pypdf PyPDF2 pymupdf
!pip install -q sentence-transformers transformers
!pip install -q faiss-cpu
!pip install -q numpy pandas

# Install LlamaIndex packages for enhanced document processing
!pip install -q llama-index
!pip install -q llama-index-readers-file
!pip install -q llama-index-embeddings-huggingface
!pip install -q llama-index-vector-stores-faiss

# Install and start Ollama (open-source, runs mistral locally — no API key needed)
!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.6/320.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.5/164.5 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 14.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the

In [ ]:
import gradio as gr
from gradio_pdf import PDF
import fitz  # PyMuPDF
from PyPDF2 import PdfReader
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
import requests
import subprocess, time
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
import json
from datetime import datetime
import hashlib

# LlamaIndex imports for enhanced document processing
from llama_index.core import Document, VectorStoreIndex, StorageContext
from llama_index.core.schema import TextNode
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.vector_stores import MetadataFilters, MetadataFilter, FilterOperator

# Start Ollama server in the background and pull the open-source Mistral model
ollama_process = subprocess.Popen(["ollama", "serve"])
time.sleep(5)
!ollama pull mistral

MISTRAL_MODEL = "mistral"

def call_mistral(prompt: str, temperature: float = 0.0) -> str:
    """Send a prompt to the local open-source Mistral model via Ollama."""
    resp = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": MISTRAL_MODEL,
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": temperature},
        },
        timeout=120,
    )
    resp.raise_for_status()
    return resp.json()["response"].strip()

# Initialize embedding models (both for compatibility)
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
llama_embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

print("Imports and configuration complete.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Imports and configuration complete.


In [ ]:
def classify_document_type(text: str, max_length: int = 1500) -> str:
    text_sample = text[:max_length] if len(text) > max_length else text
    prompt = f"""You are a pharmaceutical document classifier. ...""" # unchanged prompt body

    try:
        response_text = call_mistral(prompt)
        return clean_doc_type(response_text)
    except Exception as e:
        print(f"Classification error: {e}")
        return 'Other'

In [ ]:
def detect_document_boundary(prev_text: str, curr_text: str,
                            current_doc_type: str = None) -> bool:
    if not prev_text or not curr_text:
        return False
    prev_sample = prev_text[-500:] if len(prev_text) > 500 else prev_text
    curr_sample = curr_text[:500] if len(curr_text) > 500 else curr_text
    prompt = f"""Determine if these two pages are from the SAME pharmaceutical document. ...""" # unchanged prompt body

    try:
        response_text = call_mistral(prompt)
        return response_text.strip().lower().startswith('yes')
    except Exception as e:
        print(f"Boundary detection error: {e}")
        return True

In [ ]:
def predict_query_document_type(query: str) -> Tuple[str, float]:
    prompt = f"""Analyze this query and predict which pharmaceutical document type
would most likely contain the answer.
...
Respond in JSON format:
{{"type": "DocumentType", "confidence": 0.85}}

Confidence should be between 0.0 and 1.0"""

    try:
        response_text = call_mistral(prompt)
        cleaned = response_text.strip()
        if cleaned.startswith("```"):
            cleaned = cleaned.strip("`").replace("json", "", 1).strip()
        result = json.loads(cleaned)
        predicted = result.get("type", "Other")
        confidence = result.get("confidence", 0.5)
        return clean_doc_type(predicted), confidence
    except Exception as e:
        print(f"Query routing error: {e}")
        return "Other", 0.0

In [ ]:
def clean_doc_type(doc_type: str) -> str:
    """Normalizes document type strings."""
    doc_type = doc_type.strip().replace('_', ' ').replace('-', ' ').title()
    if ' ' in doc_type:
        return ''.join([s.capitalize() for s in doc_type.split(' ')]).replace('Of', 'of')
    return doc_type

In [ ]:
def generate_answer(prompt, retrieved_chunks, sources):
    try:
        answer = call_mistral(prompt)
        avg_score = sum(s for _, s in retrieved_chunks) / len(retrieved_chunks)
        return {
            'answer': answer,
            'sources': sources,
            'confidence': avg_score,
            'chunks_used': len(retrieved_chunks)
        }
    except Exception as e:
        print(f"Answer generation error: {e}")
        return {
            'answer': f"Error generating answer: {str(e)}",
            'sources': sources,
            'confidence': 0.0
        }

In [ ]:
@dataclass
class DocumentInfo:
    doc_id: str
    doc_type: str
    page_numbers: List[int]
    text_chunks: List[str]
    chunk_embeddings: Optional[np.ndarray] = None
    llama_nodes: Optional[List[TextNode]] = None

class EnhancedDocumentStore:
    def __init__(self):
        self.documents_by_type: Dict[str, List[DocumentInfo]] = {}
        self.faiss_indexes: Dict[str, faiss.IndexFlatL2] = {}
        self.llama_indexes: Dict[str, VectorStoreIndex] = {}
        self.all_doc_ids: set = set()
        self.is_ready = False
        self.processing_stats = {
            'filename': None,
            'total_pages': 0,
            'documents_found': 0,
            'total_chunks': 0,
            'document_types': [],
            'processing_time': 0
        }

    def _chunk_document_llama_index(self, doc_text: str, doc_id: str, doc_type: str, page_num: int) -> List[TextNode]:
        """Chunks document text into nodes using LlamaIndex's SentenceSplitter."""
        splitter = SentenceSplitter(chunk_size=512, chunk_overlap=20)
        nodes = splitter.get_nodes_from_documents(
            [Document(text=doc_text, metadata={'doc_id': doc_id, 'doc_type': doc_type, 'page_num': page_num})]
        )
        for node in nodes:
            node.embedding = llama_embed_model.get_text_embedding(node.get_content())
        return nodes

    def process_pdf(self, pdf_file_path: str, filename: str) -> Tuple[bool, Dict]:
        start_time = time.time()
        try:
            # Reset state for new PDF
            self.documents_by_type = {}
            self.faiss_indexes = {}
            self.llama_indexes = {}
            self.all_doc_ids = set()
            self.is_ready = False
            self.processing_stats = {
                'filename': filename,
                'total_pages': 0,
                'documents_found': 0,
                'total_chunks': 0,
                'document_types': [],
                'processing_time': 0
            }

            reader = PdfReader(pdf_file_path)
            num_pages = len(reader.pages)
            self.processing_stats['total_pages'] = num_pages

            doc_id_counter = 0
            current_doc_type = None
            current_doc_pages = []
            current_doc_text_buffer = []
            current_doc_chunks = []
            current_doc_embeddings = []
            current_doc_llama_nodes = []

            def _finalize_current_doc():
                nonlocal current_doc_type, current_doc_pages, current_doc_text_buffer, doc_id_counter
                nonlocal current_doc_chunks, current_doc_embeddings, current_doc_llama_nodes

                if not current_doc_text_buffer:
                    return

                doc_text = " ".join(current_doc_text_buffer)
                doc_id = hashlib.md5(f"{filename}-{doc_id_counter}".encode()).hexdigest()
                doc_id_counter += 1
                self.all_doc_ids.add(doc_id)

                # LlamaIndex chunking and embedding
                llama_nodes = self._chunk_document_llama_index(doc_text, doc_id, current_doc_type, current_doc_pages[0])
                node_texts = [node.get_content() for node in llama_nodes]
                node_embeddings = np.array([node.embedding for node in llama_nodes])

                doc_info = DocumentInfo(
                    doc_id=doc_id,
                    doc_type=current_doc_type,
                    page_numbers=current_doc_pages,
                    text_chunks=node_texts,
                    chunk_embeddings=node_embeddings,
                    llama_nodes=llama_nodes
                )

                if current_doc_type not in self.documents_by_type:
                    self.documents_by_type[current_doc_type] = []
                self.documents_by_type[current_doc_type].append(doc_info)

                # Prepare FAISS index for this type if it doesn't exist
                if current_doc_type not in self.faiss_indexes:
                    d = node_embeddings.shape[1] # Embedding dimension
                    self.faiss_indexes[current_doc_type] = faiss.IndexFlatL2(d)
                # Add embeddings to FAISS index
                self.faiss_indexes[current_doc_type].add(node_embeddings)

                # Prepare LlamaIndex VectorStoreIndex for this type if it doesn't exist
                if current_doc_type not in self.llama_indexes:
                    self.llama_indexes[current_doc_type] = VectorStoreIndex(
                        nodes=[], embed_model=llama_embed_model
                    )
                # Add nodes to LlamaIndex
                self.llama_indexes[current_doc_type].insert_nodes(llama_nodes)

                self.processing_stats['total_chunks'] += len(node_texts)
                if current_doc_type not in self.processing_stats['document_types']:
                    self.processing_stats['document_types'].append(current_doc_type)
                self.processing_stats['documents_found'] = len(self.documents_by_type)

                # Reset for next document
                current_doc_type = None
                current_doc_pages = []
                current_doc_text_buffer = []
                current_doc_chunks = []
                current_doc_embeddings = []
                current_doc_llama_nodes = []

            prev_page_text = ""
            for i in range(num_pages):
                page_text = reader.pages[i].extract_text() or ""
                doc = fitz.open(pdf_file_path)
                page_text_fitz = doc.load_page(i).get_text()
                doc.close()
                # Prefer PyMuPDF for more reliable text extraction
                page_text = page_text_fitz.strip()

                if not page_text:
                    prev_page_text = ""
                    continue

                # Initial classification for the first page or after a boundary
                if current_doc_type is None:
                    current_doc_type = classify_document_type(page_text)

                is_boundary = detect_document_boundary(
                    prev_page_text, page_text, current_doc_type
                )

                if is_boundary and current_doc_text_buffer:
                    _finalize_current_doc()
                    # Re-classify the new document's type
                    current_doc_type = classify_document_type(page_text)

                current_doc_pages.append(i + 1)
                current_doc_text_buffer.append(page_text)
                prev_page_text = page_text

            _finalize_current_doc() # Finalize the last document

            self.is_ready = True
            end_time = time.time()
            self.processing_stats['processing_time'] = f"{end_time - start_time:.2f} seconds"
            return True, self.processing_stats

        except Exception as e:
            self.is_ready = False
            end_time = time.time()
            self.processing_stats['processing_time'] = f"{end_time - start_time:.2f} seconds"
            self.processing_stats['error'] = str(e)
            return False, self.processing_stats

    def get_document_structure(self) -> List[Dict]:
        """Returns a summary of the processed documents."""
        structure = []
        for doc_type, docs in self.documents_by_type.items():
            for doc_info in docs:
                structure.append({
                    'type': doc_type,
                    'doc_id': doc_info.doc_id,
                    'pages': f"{min(doc_info.page_numbers)}-{max(doc_info.page_numbers)}",
                    'chunks': len(doc_info.text_chunks)
                })
        return structure

    def query(self, query: str, filter_type: Optional[str] = None, auto_route: bool = False, k: int = 4) -> Dict:
        """Queries the document store and returns an answer and sources."""
        if not self.is_ready:
            return {
                'answer': "Document store not ready. Please process a PDF first.",
                'sources': [],
                'confidence': 0.0,
                'filter_used': 'None'
            }

        actual_filter = filter_type

        if auto_route and filter_type is None:
            predicted_type, confidence = predict_query_document_type(query)
            if confidence > 0.6: # Threshold for auto-routing
                actual_filter = predicted_type
                # Fallback to general if predicted type isn't indexed
                if actual_filter not in self.documents_by_type and self.documents_by_type:
                    actual_filter = list(self.documents_by_type.keys())[0] # Take the first available type
            else:
                actual_filter = None # No strong prediction, query all

        # Prepare retrieval query engine(s)
        retrieved_chunks_with_scores = []
        sources = []

        if actual_filter and actual_filter in self.llama_indexes:
            # Use LlamaIndex query engine for specific types
            query_engine = self.llama_indexes[actual_filter].as_query_engine(
                similarity_top_k=k,
                filters=MetadataFilters(
                    filters=[MetadataFilter(key='doc_type', value=actual_filter, operator=FilterOperator.EQ)]
                )
            )
            li_response = query_engine.query(query)
            for node_with_score in li_response.source_nodes:
                node = node_with_score.node
                score = node_with_score.score
                retrieved_chunks_with_scores.append((node.get_content(), score))
                sources.append({
                    'doc_type': node.metadata.get('doc_type', 'Unknown'),
                    'pages': f"{node.metadata.get('page_num', 'N/A')}",
                    'relevance': f"{score:.2f}"
                })
        elif not actual_filter: # Query across all available documents using FAISS for now
            # Embed the query
            query_embedding = embed_model.encode(query)
            query_embedding = np.array([query_embedding])

            # Aggregate all chunks and embeddings for a global search (can be slow for large docs)
            all_embeddings = []
            all_chunks = []
            all_sources_map = {}

            for doc_type, docs_list in self.documents_by_type.items():
                for doc_info in docs_list:
                    if doc_info.chunk_embeddings is not None:
                        all_embeddings.append(doc_info.chunk_embeddings)
                        all_chunks.extend(doc_info.text_chunks)
                        for idx, chunk in enumerate(doc_info.text_chunks):
                            all_sources_map[len(all_chunks) - len(doc_info.text_chunks) + idx] = {
                                'doc_type': doc_type,
                                'pages': f"{min(doc_info.page_numbers)}-{max(doc_info.page_numbers)}"
                            }
            if not all_embeddings:
                 return {
                    'answer': "No documents to query. Please process a PDF first.",
                    'sources': [],
                    'confidence': 0.0,
                    'filter_used': 'None'
                }

            combined_embeddings = np.vstack(all_embeddings)
            d = combined_embeddings.shape[1]
            faiss_index_all = faiss.IndexFlatL2(d)
            faiss_index_all.add(combined_embeddings)

            # Search FAISS index
            distances, indices = faiss_index_all.search(query_embedding, k)

            # Retrieve corresponding chunks and scores
            for i, idx in enumerate(indices[0]):
                if idx < len(all_chunks):
                    score = 1 - distances[0][i] / (distances[0][0] + 1e-6) # Normalize score
                    retrieved_chunks_with_scores.append((all_chunks[idx], score))
                    src_info = all_sources_map.get(idx, {'doc_type': 'Unknown', 'pages': 'N/A'})
                    sources.append({
                        'doc_type': src_info['doc_type'],
                        'pages': src_info['pages'],
                        'relevance': f"{score:.2f}"
                    })

        # Sort chunks by score (highest first) and take the top k
        retrieved_chunks_with_scores.sort(key=lambda x: x[1], reverse=True)
        top_k_chunks = retrieved_chunks_with_scores[:k]
        if not top_k_chunks:
            return {
                'answer': "Could not find relevant information in the documents.",
                'sources': [],
                'confidence': 0.0,
                'filter_used': actual_filter if actual_filter else 'All'
            }

        context_text = "\n\n".join([chunk for chunk, _ in top_k_chunks])

        # Construct prompt for Mistral
        rag_prompt = f"""You are a helpful pharmaceutical assistant. Answer the question based on the provided documents. If the answer is not in the documents, state that you cannot find the answer.

Question: {query}

Documents:
{context_text}

Answer:"""

        # Generate answer using Mistral
        answer_response = generate_answer(rag_prompt, top_k_chunks, sources)
        answer_response['filter_used'] = actual_filter if actual_filter else 'All'
        return answer_response

In [ ]:
# Global store instance
doc_store = EnhancedDocumentStore()

def process_pdf_handler(pdf_file):
    """Handle PDF upload and processing."""
    if pdf_file is None:
        return "Please upload a PDF file", None, gr.update(choices=["All"])

    # Process the PDF
    success, stats = doc_store.process_pdf(pdf_file,
                                          filename=pdf_file.split('/')[-1] if isinstance(pdf_file, str) else
getattr(pdf_file, 'name', 'pharma-blob-sample.pdf'))

    if success:
        # Prepare status message
        status_msg = f"""
**Successfully Processed:**
- File: {stats['filename']}
- Pages: {stats['total_pages']}
- Documents Found: {stats['documents_found']}
- Chunks Created: {stats['total_chunks']}
- Types: {', '.join(stats['document_types'])}
- Time: {stats['processing_time']}
"""

        # Get document structure for display
        structure = doc_store.get_document_structure()
        structure_display = "\n".join([
            f"- **{doc['type']}** (Pages {doc['pages']}): {doc['chunks']} chunks"
            for doc in structure
        ])

        # Update filter choices
        doc_types = ["All"] + stats['document_types']

        return status_msg, structure_display, gr.update(choices=doc_types, value="All")
    else:
        return f"Error: {stats.get('error', 'Unknown error')}", None, gr.update(choices=["All"])

def chat_handler(message, history, doc_filter, auto_route, num_chunks):
    """Handle chat interactions."""
    if not doc_store.is_ready:
        response = "Please upload and process a pharmaceutical PDF document first."
        return history + [{"role": "user", "content": message}, {"role": "assistant", "content": response}]

    # Query the document store
    filter_type = None if doc_filter == "All" else doc_filter
    result = doc_store.query(
        message,
        filter_type=filter_type,
        auto_route=auto_route and filter_type is None,
        k=num_chunks
    )

    # Format response with sources
    response = f"{result['answer']}\n\n"

    if result['sources']:
        response += "**Sources:**\n"
        for src in result['sources']:
            response += f"- {src['doc_type']} (Pages {src['pages']}) - Relevance: {src['relevance']}\n"

    response += f"\n*Confidence: {result['confidence']:.1%} | Filter: {result['filter_used']}*"

    return history + [{"role": "user", "content": message}, {"role": "assistant", "content": response}]

def create_interface():
    """Create the Gradio interface for pharmaceutical document Q&A."""

    with gr.Blocks(title="Pharmaceutical Document Q&A System") as demo:
        gr.Markdown("""
        # Pharmaceutical Document Q&A System
        ### Intelligent Multi-Document Analysis with Advanced RAG Pipeline
        Upload a pharmaceutical blob PDF (e.g. pharma-blob-sample.pdf) to identify
        document types, build a searchable index, and ask questions in natural language.
        """)

        with gr.Row():
            # Left side - PDF upload
            with gr.Column(scale=2):
                pdf_input = gr.File(
                    label="Upload Pharmaceutical PDF",
                    file_types=[".pdf"],
                    type="filepath"
                )

                with gr.Row():
                    process_btn = gr.Button(
                        "Process Document",
                        variant="primary",
                        size="lg",
                        scale=2
                    )
                    clear_all_btn = gr.Button(
                        "Clear All",
                        variant="secondary",
                        size="lg",
                        scale=1
                    )

            # Middle - Document info and settings
            with gr.Column(scale=1):
                gr.Markdown("### Document Info")
                status_output = gr.Markdown(
                    value="Waiting for PDF upload..."
                )

                structure_output = gr.Markdown(
                    value="",
                    label="Document Structure"
                )

                gr.Markdown("### Retrieval Settings")

                doc_filter = gr.Dropdown(
                    choices=["All"],
                    value="All",
                    label="Document Type Filter",
                    info="Filter search to a specific pharmaceutical document type"
                )

                auto_route = gr.Checkbox(
                    value=True,
                    label="Auto-Route Queries",
                    info="Automatically detect the most relevant document type"
                )

                num_chunks = gr.Slider(
                    minimum=1,
                    maximum=10,
                    value=4,
                    step=1,
                    label="Chunks to Retrieve"
                )

            # Right side - Chat interface
            with gr.Column(scale=2):
                gr.Markdown("### Ask Questions")
                chatbot = gr.Chatbot(
                    label="Conversation",
                    height=500,
                    elem_id="chatbot",
                    show_label=False,
                )

                with gr.Row():
                    msg_input = gr.Textbox(
                        label="Ask a question",
                        placeholder="e.g., What is the lot number? What sterilization method was used?",
                        scale=4,
                        show_label=False
                    )
                    send_btn = gr.Button("Send", scale=1, variant="primary")

                with gr.Row():
                    clear_chat_btn = gr.Button("Clear Chat", size="sm", scale=1)
                    example_btn1 = gr.Button("Summarise this document", size="sm", scale=1)
                    example_btn2 = gr.Button("Find lot numbers", size="sm", scale=1)

        # Status bar at the bottom
        with gr.Row():
            status_bar = gr.Markdown(
                value="**Status:** Ready | **Documents:** 0 | **Chunks:** 0",
                elem_id="status_bar"
            )

        # Event handlers
        def update_status_bar():
            """Update the status bar with current statistics."""
            if doc_store.is_ready:
                stats = doc_store.processing_stats
                return (
                    f"**Status:** Ready | "
                    f"**Documents:** {stats.get('documents_found', 0)} | "
                    f"**Chunks:** {stats.get('total_chunks', 0)}"
                )
            return "**Status:** Ready | **Documents:** 0 | **Chunks:** 0"

        def clear_all():
            """Clear everything and reset the interface."""
            global doc_store
            doc_store = EnhancedDocumentStore()
            return (
                None,  # pdf_input
                "Waiting for PDF upload...",  # status_output
                "",  # structure_output
                gr.update(choices=["All"], value="All"),  # doc_filter
                [],  # chatbot
                "",  # msg_input
                update_status_bar()  # status_bar
            )

        # Process PDF handler with status bar update
        def process_pdf_with_status(pdf_file):
            status, structure, filter_update = process_pdf_handler(pdf_file)
            status_bar_text = update_status_bar()
            return status, structure, filter_update, status_bar_text

        # Chat handler with status bar update
        def chat_with_status(message, history, doc_filter, auto_route, num_chunks):
            new_history = chat_handler(message, history, doc_filter, auto_route, num_chunks)
            status_bar_text = update_status_bar()
            return new_history, status_bar_text

        # Example question handlers
        def ask_summary(history):
            return chat_handler(
                "Can you provide a summary of the main points in this document?",
                history, doc_filter.value, auto_route.value, num_chunks.value
            )

        def ask_lot_numbers(history):
            return chat_handler(
                "What lot numbers or batch numbers are mentioned in these documents?",
                history, doc_filter.value, auto_route.value, num_chunks.value
            )

        # Wire up all the events
        process_btn.click(
            fn=process_pdf_with_status,
            inputs=[pdf_input],
            outputs=[status_output, structure_output, doc_filter, status_bar]
        )

        clear_all_btn.click(
            fn=clear_all,
            outputs=[pdf_input, status_output, structure_output, doc_filter,
                    chatbot, msg_input, status_bar]
        )

        # Chat interactions
        msg_input.submit(
            fn=chat_with_status,
            inputs=[msg_input, chatbot, doc_filter, auto_route, num_chunks],
            outputs=[chatbot, status_bar]
        ).then(
            lambda: "",
            outputs=[msg_input]
        )

        send_btn.click(
            fn=chat_with_status,
            inputs=[msg_input, chatbot, doc_filter, auto_route, num_chunks],
            outputs=[chatbot, status_bar]
        ).then(
            lambda: "",
            outputs=[msg_input]
        )

        clear_chat_btn.click(
            lambda: [],
            outputs=[chatbot]
        )

        example_btn1.click(
            fn=ask_summary,
            inputs=[chatbot],
            outputs=[chatbot]
        ).then(
            fn=update_status_bar,
            outputs=[status_bar]
        )

        example_btn2.click(
            fn=ask_lot_numbers,
            inputs=[chatbot],
            outputs=[chatbot]
        ).then(
            fn=update_status_bar,
            outputs=[status_bar]
        )

        # Auto-process when PDF is uploaded
        pdf_input.change(
            fn=process_pdf_with_status,
            inputs=[pdf_input],
            outputs=[status_output, structure_output, doc_filter, status_bar]
        )

    return demo

In [ ]:
demo = create_interface()
demo.launch(share=True, debug=True, theme=gr.themes.Soft())

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7c540c980b11629ecd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Classification error: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=120)
Boundary detection error: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=120)
Classification error: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=120)
Boundary detection error: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=120)
Classification error: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=120)
Boundary detection error: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=120)
Classification error: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=120)
Boundary detection error: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=120)
Classification error: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=120)
Boundary detection er